# **First Homework Assignment**

In [49]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [50]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0:
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)

# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)

In [51]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Sequential vs Parallelized**

In [55]:
# Specifiying the parameters of the analysis.
num_cores = ["seq", 1, 2, 4, 8, 16]
repetitions = 5

_csv = pd.read_csv("results_parallel.csv")
_csv.columns = [str(c).strip().strip('"') for c in _csv.columns]
_csv["Time_sec"] = pd.to_numeric(
    _csv["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
    errors="coerce",
)
_means = _csv.groupby(["Image", "Method", "Threads"])["Time_sec"].mean()
infiles = sorted(_csv["Image"].unique())

In [56]:
# Parsing metrics equal for all iterations.
df = pd.DataFrame(columns=num_cores, index=infiles)
df.attrs['name'] = f"Time averages over 5 repetitions"
for N in num_cores:
    method = "seam-carving" if N == "seq" else "seam-carving-omp"
    threads = 1 if N == "seq" else int(N)
    for filename in infiles:
        key = (filename, method, threads)
        df.loc[filename, N] = _means[key] if key in _means.index else np.nan

    # Miss ratio in percents.
    df.loc["Mean"] = df.mean(axis=0)

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([df], axis=1)

Maximal values per row are displayed in red, while minimal are displayed in green.


,seq,1,2,4,8,16
1024x768.png,1.627,2.975,1.551,0.833,0.626,0.628
1920x1200.png,4.65,8.796,4.499,2.362,1.513,1.224
3840x2160.png,21.556,36.28,18.509,9.594,5.533,3.732
720x480.png,0.673,1.284,0.68,0.379,0.311,0.348
7680x4320.png,92.397,156.381,78.988,40.655,22.315,13.553
test.png,0.488,0.925,0.507,0.291,0.267,
valve.png,0.521,0.988,0.535,0.3,0.268,0.318
Mean,17.416,29.661,15.038,7.773,4.405,3.3


In [57]:
# Average speed-up for 16 cores: sequential time / parallel time, averaged over images.
_img_rows = [i for i in df.index if i != "Mean"]
_speedup_16 = df.loc[_img_rows, "seq"] / df.loc[_img_rows, 16]
avg_speedup_16 = _speedup_16.mean(skipna=True)
print(f"Average speed-up (16 cores vs seq.): {avg_speedup_16:.3f}x")

_speedup_7680 = df.loc["7680x4320.png", "seq"] / df.loc["7680x4320.png", 16]
print(f"Speed-up (16 cores vs seq.) for 7680x4320.png: {_speedup_7680:.3f}x")

Average speed-up (16 cores vs seq.): 3.759x
Speed-up (16 cores vs seq.) for 7680x4320.png: 6.817x


## **Dynamic Thread Allocation**

In [58]:
def best_time_per_image(csv_df):
    """Mean time per (image, E, DP, R), then the row with minimum time for each image."""
    g = csv_df.groupby(["Image", "E_const", "DP_const", "R_const"], as_index=False)["Time_sec"].mean()
    pick = g.loc[g.groupby("Image")["Time_sec"].idxmin()].reset_index(drop=True)
    pick = pick.rename(
        columns={
            "Image": "image",
            "Time_sec": "time",
            "E_const": "C1",
            "DP_const": "C2",
            "R_const": "C3",
        }
    )
    return pick[
        [
            "image",
            "time",
            "C1",
            "C2",
            "C3",
        ]
    ].sort_values("image")

In [ ]:
# Specifiying the parameters of the analysis.
repetitions = 1

_csv_opt = pd.read_csv("results_optimized-32.csv")
_csv_opt.columns = [str(c).strip().strip('"') for c in _csv_opt.columns]
_csv_opt["Time_sec"] = pd.to_numeric(
    _csv_opt["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
    errors="coerce",
)
infiles_opt = sorted(_csv_opt["Image"].unique())

In [60]:
# Parsing metrics equal for all iterations.
df_opt = best_time_per_image(_csv_opt)
df_opt.attrs["name"] = f"Best heuristic per image on "

display_list([df_opt.set_index("image")], index=True)

,time,C1,C2,C3
image,,,,
1024x768.png,0.471,500,1000,100
1920x1200.png,0.778,1000,500,100
3840x2160.png,2.502,500,1000,100
720x480.png,0.333,1000,500,100
7680x4320.png,9.154,500,1000,100
test.png,0.246,500,5000,5000
valve.png,0.271,5000,1000,500


In [61]:
# Specifiying the parameters of the analysis.
repetitions = 1

_csv_opt = pd.read_csv("results_optimized-16.csv")
_csv_opt.columns = [str(c).strip().strip('"') for c in _csv_opt.columns]
_csv_opt["Time_sec"] = pd.to_numeric(
    _csv_opt["Time"].astype(str).str.strip().str.replace(r"\s*s$", "", regex=True),
    errors="coerce",
)
infiles_opt = sorted(_csv_opt["Image"].unique())

In [62]:
# Parsing metrics equal for all iterations.
df_opt = best_time_per_image(_csv_opt)
df_opt.attrs["name"] = f"Best heuristic per image on "

display_list([df_opt.set_index("image")], index=True)

,time,C1,C2,C3
image,,,,
1024x768.png,0.412,5000,1000,100
1920x1200.png,0.894,5000,500,100
3840x2160.png,3.477,5000,1000,100
720x480.png,0.252,1000,1000,100
7680x4320.png,14.008,5000,1000,500
valve.png,0.212,5000,1000,100
